### Import Required Dependencies

In [6]:
# --- Imports ---
import pandas as pd
import numpy as np
import warnings
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import re


from pathlib import Path
from functools import reduce
from sklearn.preprocessing import MinMaxScaler
from collections import Counter

# --- Setup ---
sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
warnings.filterwarnings('ignore')

# --- Paths ---
data_dir = Path("data")
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)

image_dir = Path("outputs/images")
image_dir.mkdir(parents=True, exist_ok=True)

# --- Logging ---
logging.basicConfig(
    filename=log_dir / "project.log",
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    filemode='w'
)
console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("[%(levelname)s] %(message)s")
console.setFormatter(formatter)
logging.getLogger().addHandler(console)

print("✅ Environment ready. Paths and logging configured.")



✅ Environment ready. Paths and logging configured.


# Section 2 Load and Process Dataset (national and Arkansas)

## Section 2A: Load utility functions

In [7]:
# Section 2A: Load utility functions and national datasets ----

# Import custom utility functions
from utils import (
    standardize_column_names,
    extract_key_indicators,
    filter_to_county_level,
    log_duplicate_attributes,
    clean_and_extract_year,
    nca_counties
)

data_dir = Path("data")
complete_dir = data_dir / "complete_sets"

complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

# Load, clean, and extract year from each dataset
complete_data = {}

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        logging.info(f"📌 {key} columns after cleaning: {df.columns.tolist()}")
        df = filter_to_county_level(df)
        df = clean_and_extract_year(df)

        # Rename fields if necessary
        df.rename(columns={'area_name': 'county'}, inplace=True)

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f"✅ Loaded {filename}: {df.shape[0]} rows")

    except Exception as e:
        logging.error(f"❌ Failed to load {filename}: {e}")


# 4. Confirmation message
logging.info("🧠 Utility functions from utils.py loaded successfully.")


[INFO] 📌 edu columns after cleaning: ['ïfips_code', 'state', 'area_name', 'attribute', 'value']
[INFO] 📌 edu columns after cleaning: ['ïfips_code', 'state', 'area_name', 'attribute', 'value']
[WARNING] ⚠️ edu has duplicate county-attribute pairs.
[WARNING] ⚠️ edu has duplicate county-attribute pairs.
[INFO] ✅ Loaded Education2023.csv: 169245 rows
[INFO] ✅ Loaded Education2023.csv: 169245 rows
[INFO] 📌 pop columns after cleaning: ['fipstxt', 'state', 'area_name', 'attribute', 'value']
[INFO] 📌 pop columns after cleaning: ['fipstxt', 'state', 'area_name', 'attribute', 'value']
[WARNING] ⚠️ pop has duplicate county-attribute pairs.
[WARNING] ⚠️ pop has duplicate county-attribute pairs.
[INFO] ✅ Loaded PopulationEstimates.csv: 205108 rows
[INFO] ✅ Loaded PopulationEstimates.csv: 205108 rows
[INFO] 📌 poverty columns after cleaning: ['ïfips_code', 'state', 'area_name', 'attribute', 'value']
[INFO] 📌 poverty columns after cleaning: ['ïfips_code', 'state', 'area_name', 'attribute', 'value']
[W

## Section 2B: Load and process national datasets

In [8]:
# ✅ Section 2B: Load and process national datasets

complete_dir = data_dir / "complete_sets"
complete_files = {
    "edu": "Education2023.csv",
    "pop": "PopulationEstimates.csv",
    "poverty": "Poverty2023.csv",
    "unemp": "Unemployment2023.csv"
}

state_lookup = None

# 👇 Choose your analysis year here
selected_year = '2022'

for key, filename in complete_files.items():
    path = complete_dir / filename
    try:
        # ✅ Load and standardize
        df = pd.read_csv(path, encoding='cp1252')
        df = standardize_column_names(df)
        df = filter_to_county_level(df)
        df.rename(columns={'area_name': 'county', 'fips_code': 'fips', 'fipstxt': 'fips'}, inplace=True)
        df['county'] = df['county'].str.strip().str.lower()

        # ✅ Extract year from attribute
        if 'attribute' in df.columns:
            df = clean_and_extract_year(df)

            # ✅ Only filter datasets where year tagging applies
            if key in ['pop', 'poverty', 'unemp']:
                df = df[df['attribute_year'] == selected_year]

        # ✅ Save state info once from education
        if key == 'edu':
            state_lookup = df[['county', 'state']].drop_duplicates().copy()
            state_lookup['county'] = state_lookup['county'].str.lower().str.strip()
            state_lookup['state'] = state_lookup['state'].str.upper().str.strip()

        log_duplicate_attributes(df, key)
        complete_data[key] = df
        logging.info(f"✅ Loaded and processed {filename} with year filter: {selected_year if key != 'edu' else 'N/A'}")

    except Exception as e:
        logging.error(f"❌ Failed to load {filename}: {e}")


[WARNING] ⚠️ edu has duplicate county-attribute pairs.
[WARNING] ⚠️ edu has duplicate county-attribute pairs.
[INFO] ✅ Loaded and processed Education2023.csv with year filter: N/A
[INFO] ✅ Loaded and processed Education2023.csv with year filter: N/A
[WARNING] ⚠️ pop has duplicate county-attribute pairs.
[WARNING] ⚠️ pop has duplicate county-attribute pairs.
[INFO] ✅ Loaded and processed PopulationEstimates.csv with year filter: 2022
[INFO] ✅ Loaded and processed PopulationEstimates.csv with year filter: 2022
[INFO] ✅ Loaded and processed Poverty2023.csv with year filter: 2022
[INFO] ✅ Loaded and processed Poverty2023.csv with year filter: 2022
[INFO] ✅ Loaded and processed Unemployment2023.csv with year filter: 2022
[INFO] ✅ Loaded and processed Unemployment2023.csv with year filter: 2022


### Section 2B.1 Determine most common year across all datasets

In [9]:
## 📊 Section 2B.1: Discover Common Year Across Datasets

def get_attribute_year_counts(df):
    """Counts how often each extracted attribute_year appears."""
    if 'attribute' in df.columns:
        df = clean_and_extract_year(df)
        return Counter(df['attribute_year'].dropna().astype(str))
    return {}

year_summary = {}

# Loop through each dataset and extract year counts
for key, df in complete_data.items():
    year_counts = get_attribute_year_counts(df)
    year_summary[key] = year_counts

# Display results
print("📊 Year coverage by dataset:")
all_years = set()

for key, counts in year_summary.items():
    print(f"\n🔹 {key.upper()}:")
    if counts:
        for year, count in sorted(counts.items()):
            print(f"  {year}: {count}")
            all_years.add(year)
    else:
        print("  ❌ No attribute_year values found.")

# Find common years across all datasets that have year values
datasets_with_years = [set(c.keys()) for c in year_summary.values() if c]
common_years = set.intersection(*datasets_with_years) if datasets_with_years else set()



print("\n✅ Common years across all datasets with usable attribute_year:", sorted(common_years))


📊 Year coverage by dataset:

🔹 EDU:
  1970: 25488
  1980: 26136
  1990: 26167
  2000: 26176
  2012: 26192
  2013: 6442
  2023: 29422
  2024: 3222

🔹 POP:
  2022: 50397

🔹 POVERTY:
  ❌ No attribute_year values found.

🔹 UNEMP:
  2022: 19171

✅ Common years across all datasets with usable attribute_year: []


### Section 2B.2: Merge cleaned datasets into df_full

In [10]:
# --- Section 2B.2: Merge cleaned datasets into df_full ---
merge_keys = ['fips', 'county']

# Start with education
df_full = complete_data['edu']

# Merge in the rest (poverty, unemp, pop)
for key in ['poverty', 'unemp', 'pop']:
    df_full = pd.merge(df_full, complete_data[key], on=merge_keys, how='outer', suffixes=('', f'_{key}'))

# Add state info back if necessary
if state_lookup is not None:
    df_full = pd.merge(df_full, state_lookup, on='county', how='left')

# Ensure 'state' column is uppercase
df_full['state'] = df_full['state'].str.upper()


KeyError: 'fips'

## Section 2C: Subset Arkansas and NCA Counties

In [ ]:
# Section 2C: Subset Arkansas and NCA counties

df_ar = df_full[df_full['state'] == 'AR'].copy()
df_ar = df_ar.reset_index()
df_ar['county'] = df_ar['county'].str.replace(" county", "", regex=False).str.replace(", ar", "", regex=False).str.strip()
df_ar.set_index('county', inplace=True)

df_nca = df_ar[df_ar.index.isin(nca_counties)].copy()
df_ar.to_csv(output_dir / "arkansas_counties.csv")
df_nca.to_csv(output_dir / "nca_counties.csv")

logging.info(f"📌 Arkansas counties: {df_ar.shape[0]}")
logging.info(f"📌 NCA counties: {df_nca.shape[0]}")

df_nca.to_csv(output_dir / f"nca_dataset_cleaned_{selected_year}.csv")
logging.info(f"📁 Saved final NCA dataset to: nca_dataset_cleaned_{selected_year}.csv")

## Section 3: Exploratory Data Analysis (EDA)

In [ ]:
# SECTION 3: Exploratory Data Analysis (EDA)

# Use a working copy of the NCA subset
df = df_nca.copy()
logging.info(f"🔍 Starting EDA on NCA dataset: {df.shape[0]} counties, {df.shape[1]} features")


### Section 3.1: Inspect Dataset

In [ ]:
# 🧠 Full dataset overview
print("🔧 Info:")
display(df.info())

print("\n📊 Descriptive Stats:")
display(df.describe(include='all'))

print("\n🔍 Sample Data:")
display(df.head())


### Section 3.2 Extract and Rename Key Variables

In [ ]:
## Section 3.2: Extract and Name Key Indicator Variables

# 📂 Print all column names for inspection
print("\n📂 All column names in df_nca:")
for col in df_nca.columns:
    print(col)

# 🔍 Extract key indicators
df_nca, used_columns, year = extract_key_indicators(df_nca, min_year=2022)

# 📊 Print matched column summary
print("📅 Most common year in column names:", year)
print("📚 Education columns used:", used_columns['education'])
print("📉 Poverty columns used:", used_columns['poverty'])
print("💼 Unemployment columns used:", used_columns['unemployment'])
print("👥 Population columns used:", used_columns['population'])

# 🧮 Compute education percentages (relative to population)
df_nca['BachelorsDegreePct'] = (df_nca['BachelorsDegreeRate'] / df_nca['Population']) * 100
df_nca['HighSchoolGradPct'] = (df_nca['HighSchoolGradRate'] / df_nca['Population']) * 100

# ✅ Define clean variable set for EDA (use percentages)
variables = ['BachelorsDegreePct', 'HighSchoolGradPct', 'PovertyRate', 'UnemploymentRate', 'Population']

# 🔎 Preview the cleaned dataset
print("\n🔎 Preview of standardized indicators:")
display(df_nca[variables].head())

# 🧭 Debug: Print all matched and available columns
print("\n📚 Matched columns by indicator:")
for key, cols in used_columns.items():
    print(f"  - {key}: {cols}")

print("\n📁 All available columns:")
print(df_nca.columns.tolist())



### Section 3.3 Visualize Education Levels

In [ ]:
# Education distribution across counties
title = "Education Indicators by County"
df[education_cols].T.plot(kind='bar', figsize=(14, 6), title=title)
plt.ylabel("Percent or Count")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Standardize filename
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f" Saved: {filename}.png")

plt.show()



In [ ]:
print('unemployment_rate_2023' in df_nca.columns)  # Should return True
print(df_nca['unemployment_rate_2023'].head())     # Preview values


### Section 3.4: Distribution Plots (Histograms & KDE)

In [ ]:
# Distribution plots with title-based saving
variables = ['PovertyRate', 'UnemploymentRate', 'HighSchoolGradRate', 'BachelorsDegreeRate']

for var in variables:
    plt.figure(figsize=(8, 4))
    
    # Define plot title
    title = f"Distribution of {var}"
    
    # Plot
    sns.histplot(df[var], kde=True, bins=20)
    plt.title(title)
    plt.xlabel(var)
    plt.ylabel('Frequency')
    plt.tight_layout()
    
    # Create safe filename from title
    filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
    plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
    logging.info(f"📷 Saved: {filename}.png")
    
    plt.show()


### Section 3.5: Correlation Heatmap

In [ ]:
# Correlation matrix and heatmap
corr_vars = df[['PovertyRate', 'UnemploymentRate', 'HighSchoolGradRate', 'BachelorsDegreeRate', 'Population']]
corr_matrix = corr_vars.corr()

# Define title
title = "Correlation Between Key Indicators"

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title(title)
plt.tight_layout()

# Generate safe filename from title
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")

plt.show()


### Section 3.6: Key Relationships (Scatter Plots)

In [ ]:
# Scatter Plots of Key Relationships
# 1. Bachelor's Degree vs Poverty
title = "Bachelor's Degree Rate vs. Poverty Rate"
sns.scatterplot(x='BachelorsDegreeRate', y='PovertyRate', data=df)
plt.title(title)
plt.xlabel("Bachelor's Degree (%)")
plt.ylabel("Poverty Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()

# 2. High School Grad Rate vs Unemployment
title = "High School Grad Rate vs. Unemployment Rate"
sns.scatterplot(x='HighSchoolGradRate', y='UnemploymentRate', data=df)
plt.title(title)
plt.xlabel("High School Grad (%)")
plt.ylabel("Unemployment Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()

# 3. Population vs Poverty Rate
title = "Population vs. Poverty Rate"
sns.scatterplot(x='Population', y='PovertyRate', data=df)
plt.title(title)
plt.xlabel("Population")
plt.ylabel("Poverty Rate (%)")
plt.tight_layout()
filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
logging.info(f"📷 Saved: {filename}.png")
plt.show()


### Section 3.7: Outlier Detection with BoxPlots

In [ ]:
# --- Boxplots for Outlier Detection (with title-based save) ---
for var in variables:
    plt.figure(figsize=(8, 4))

    # Define title and filename
    title = f"Boxplot of {var}"
    sns.boxplot(x=df[var])
    plt.title(title)
    plt.tight_layout()

    # Standardize filename
    filename = title.lower().replace(" ", "_").replace("’", "").replace("'", "")
    plt.savefig(image_dir / f"{filename}.png", dpi=300, bbox_inches='tight')
    logging.info(f"📷 Saved: {filename}.png")

    plt.show()


### Section 3.8 ParPlot for Key Indicators

In [ ]:
# Pairplot for Key Indicators
sns.pairplot(df[variables], diag_kind='kde')
plt.suptitle("Pairwise Relationships Between Key Indicators", y=1.02)
plt.tight_layout()
plt.savefig(image_dir / "pairplot_key_indicators.png", dpi=300, bbox_inches='tight')
logging.info("📷 Saved: pairplot_key_indicators.png")
plt.show()


### Step 4: Visualize Distributions (Histograms & KDE)

### Step 5: Correlation Matrix and Heatmap

### Step 6: Scatter Plots for Key Relationships

### Step 7: Identify Outlier Counties with Boxplots

### Step 8: Log-Transform Population (Optional)
If the population is skewed: